# SGFND - Synthetic Generative Framework Neural Dynamics

This notebook compiles and runs the SGFND C project on Kaggle.
The project generates synthetic images using a custom neural dynamics engine with:
- Bridge-based feature representation (92→46 compressed bridges)
- Latent diffusion model (512-dim, 1000 timesteps)
- Tiled rendering (64×64 tiles)
- Large model streaming (30,000 bridges)
- NSFW content filtering (prompt + image analysis)

## ⚡ Quick Setup for Kaggle

**Option 1: Upload as Dataset (Recommended)**
1. Zip the `sgfnd` folder (source code, Makefile, include/)
2. Go to Kaggle → Datasets → Create New Dataset → Upload zip
3. In this notebook: Settings (⚙) → Add Data → Select your dataset
4. It will mount at `/kaggle/input/<username>/<dataset-name>/`
5. Update `DATASET_PATH` in the next cell

**Option 2: Git Clone** (if you push to GitHub)

**Option 3: Kaggle Dataset URL** (if already public)

In [ ]:
# ===== Load .env and authenticate GitHub =====
import os

def load_env(path='.env'):
    """Load .env file and return dict of key=value pairs."""
    env = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    key, val = line.split('=', 1)
                    env[key.strip()] = val.strip()
    return env

def mask_value(val):
    """Mask sensitive values (show first 4 and last 4 chars)."""
    if len(val) <= 10:
        return val[:2] + '*' * (len(val) - 4) + val[-2:]
    return val[:4] + '*' * (len(val) - 8) + val[-4:]

# Load .env from Kaggle dataset or local
env_paths = [
    '/kaggle/input/.env/.env',
    '/kaggle/working/.env',
    os.path.expanduser('~/.env'),
    '.env',
]

env = {}
for p in env_paths:
    if os.path.exists(p):
        env = load_env(p)
        print(f'Loaded .env from: {p}')
        break

if not env:
    print('No .env file found. Skipping auth.')
else:
    print('\n.env contents (masked):')
    for k, v in env.items():
        print(f'  {k} = {mask_value(v)}')

# Authenticate git with token if available
if 'github_token' in env:
    token = env['github_token']
    os.environ['GIT_ASKPASS'] = 'echo'
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    # Set remote with token for push
    print(f'\nGit authenticated with token: {mask_value(token)}')
else:
    print('\nNo github_token in .env - git push will need manual auth')

In [ ]:
# ===== CONFIGURE THIS FOR YOUR KAGGLE SETUP =====
# Option A: Mounted dataset path (after adding dataset in Settings)
DATASET_PATH = '/kaggle/input/your-username/your-dataset-name/'  # <-- CHANGE THIS

# Option B: GitHub repo (uncomment and add URL)
GIT_REPO = 'https://github.com/wippsanrinthailand80-commits/sgfnd.git'

# Option C: Direct download URL (uncomment)
# DOWNLOAD_URL = 'https://example.com/sgfnd.zip'

import os, shutil, subprocess, sys

WORK_DIR = '/kaggle/working/sgfnd'

# Clean previous
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# Copy from mounted dataset
if os.path.exists(DATASET_PATH):
    # Find sgfnd folder inside dataset
    for root, dirs, files in os.walk(DATASET_PATH):
        if 'Makefile' in files and 'src' in dirs:
            shutil.copytree(root, WORK_DIR, dirs_exist_ok=True)
            print(f'Copied from dataset: {root}')
            break
    else:
        # Maybe dataset IS the sgfnd folder
        if os.path.exists(os.path.join(DATASET_PATH, 'Makefile')):
            shutil.copytree(DATASET_PATH, WORK_DIR, dirs_exist_ok=True)
            print(f'Copied from dataset root: {DATASET_PATH}')
        else:
            print(f'⚠ Could not find sgfnd source in {DATASET_PATH}')
            print('  Check DATASET_PATH and dataset structure')

os.chdir(WORK_DIR)
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Fallback: If no dataset mounted, try git clone with token auth
if not os.path.exists('Makefile'):
    print('No local source found, attempting fallback...')
    
    # Try git clone if GIT_REPO defined
    try:
        if 'GIT_REPO' in globals() and GIT_REPO:
            # Use token for authentication if available
            if 'github_token' in env:
                auth_url = GIT_REPO.replace('https://', f'https://{env["github_token"]}@')
                subprocess.run(['git', 'clone', auth_url, '.'], check=True)
            else:
                subprocess.run(['git', 'clone', GIT_REPO, '.'], check=True)
            print('Cloned from git')
    except NameError:
        pass
    
    # Try direct download if DOWNLOAD_URL defined
    try:
        if 'DOWNLOAD_URL' in globals() and DOWNLOAD_URL:
            subprocess.run(['wget', '-q', '-O', 'sgfnd.zip', DOWNLOAD_URL], check=True)
            subprocess.run(['unzip', '-q', 'sgfnd.zip'], check=True)
            for item in os.listdir('.'):
                if os.path.isdir(item) and 'sgfnd' in item.lower():
                    for f_item in os.listdir(item):
                        shutil.move(os.path.join(item, f_item), '.')
                    os.rmdir(item)
                    break
            print('Downloaded and extracted')
    except NameError:
        pass

if not os.path.exists('Makefile'):
    raise FileNotFoundError(
        'Could not find sgfnd source. Please:\n'
        '1. Upload sgfnd folder as Kaggle Dataset\n'
        '2. Mount it in notebook Settings → Add Data\n'
        '3. Set DATASET_PATH to the mount point\n'
        '4. Or define GIT_REPO / DOWNLOAD_URL above')


In [ ]:
# Install system dependencies (Kaggle has gcc, make, but may need curl/json dev libs)
!apt-get update -qq && apt-get install -y -qq libcurl4-openssl-dev libcjson-dev 2>/dev/null | tail -5

In [ ]:
# Build the project (optimized release build)
%%bash
set -e
make clean
make
echo 'Build complete. Binary:'
ls -lh sgfnd

# Show all available options
./sgfnd --help

In [ ]:
# Run standard generation (small model, 512x512)
%%bash
./sgfnd --generate
echo 'Output files:'
ls -lh *.raw *.png 2>/dev/null

In [ ]:
# Convert raw output to PNG and display
from PIL import Image
import numpy as np

def load_raw_image(path):
    with open(path, 'rb') as f:
        header = np.fromfile(f, dtype=np.uint32, count=4)
        w, h, c, bit_depth = header
        data = np.fromfile(f, dtype=np.uint8)
        img = data.reshape(h, w, c)
        return img

img = load_raw_image('generated_cuda_image.raw')
print(f'Shape: {img.shape}, dtype: {img.dtype}, range: [{img.min()}, {img.max()}]')

# Display
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title('SGFND Generated Image (Small Model)')
plt.show()

In [ ]:
# Run tiled streaming renderer
%%bash
./sgfnd --generate --tiled
ls -lh generated_tiled.raw

In [ ]:
# Display tiled output
img_tiled = load_raw_image('generated_tiled.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img_tiled)
plt.axis('off')
plt.title('SGFND Tiled Streaming Render')
plt.show()

In [ ]:
# Run large model (30,000 bridges, streaming)
%%bash
./sgfnd --generate --large
ls -lh generated_cuda_image.raw

In [ ]:
# Display large model output
img_large = load_raw_image('generated_cuda_image.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img_large)
plt.axis('off')
plt.title('SGFND Large Model (30K Bridges)')
plt.show()

## NSFW Filter Testing

The NSFW filter has three modes:
- `disabled` (default) - no filtering
- `enabled` - checks prompts and images, logs flags
- `strict` - same as enabled but with lower default threshold

Options:
- `--nsfw MODE` - disabled|enabled|strict
- `--nsfw-thresh F` - threshold 0.0-1.0 (default: 0.5)

In [ ]:
# Test NSFW filter with small model (enabled mode)
%%bash
./sgfnd --generate --small --nsfw enabled
echo '---'
ls -lh generated_cuda_image.raw

In [ ]:
# Display small model with NSFW filter
img = load_raw_image('generated_cuda_image.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Small Model + NSFW Enabled')
plt.show()

In [ ]:
# Test NSFW filter with large model (strict mode, lower threshold)
%%bash
./sgfnd --generate --large --nsfw strict --nsfw-thresh 0.3
echo '---'
ls -lh generated_cuda_image.raw

In [ ]:
# Display large model with NSFW strict filter
img = load_raw_image('generated_cuda_image.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Large Model + NSFW Strict (thresh=0.3)')
plt.show()

In [ ]:
# Test NSFW filter with tiled renderer
%%bash
./sgfnd --generate --tiled --nsfw enabled
echo '---'
ls -lh generated_tiled.raw

In [ ]:
# Display tiled with NSFW filter
img = load_raw_image('generated_tiled.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Tiled Renderer + NSFW Enabled')
plt.show()

In [ ]:
# Combined: Large + Tiled + NSFW Strict
%%bash
./sgfnd --generate --large --tiled --nsfw strict --nsfw-thresh 0.5
echo '---'
ls -lh generated_tiled.raw generated_cuda_image.raw 2>/dev/null

In [ ]:
# Memory check with valgrind (optional, slower)
%%bash
apt-get install -y -qq valgrind 2>/dev/null
valgrind --tool=memcheck --error-exitcode=1 ./sgfnd --generate --nsfw enabled 2>&1 | tail -20

In [ ]:
# Valgrind with all features enabled
%%bash
valgrind --tool=memcheck --error-exitcode=1 ./sgfnd --generate --large --tiled --nsfw strict --nsfw-thresh 0.5 2>&1 | tail -20

In [ ]:
# Training mode example (fetch from URL, train diffusion, generate)
# Note: Requires internet access and a valid image dataset URL
%%bash
# ./sgfnd --train --url https://example.com/dataset.zip --epochs 5 --generate --nsfw enabled
echo 'Training mode available. Uncomment and provide a dataset URL to run.'

## Project Structure

```
sgfnd/
├── include/sgfnd_core.h          # Public API
├── src/
│   ├── main.c                    # CLI entry, argument parsing, NSFW integration
│   ├── bridges/bridge_engine.c   # Bridge processing & compression
│   ├── color/color_grader.c      # HSV-based color grading
│   ├── generative/
│   │   ├── latent_diffusion.c    # MLP + DDPM diffusion
│   │   └── model_training.c      # Dataset, VAE, training loop
│   ├── io/image_io.c             # Tiled image I/O, rendering
│   ├── large_model/large_model.c # 30K bridge streaming
│   ├── safety/nsfw_filter.c      # NSFW prompt/image detection & sanitization
│   └── training/training_bot.c   # Auto-fetch, augment, train
├── tools/raw_to_image.py         # Raw→PNG converter
└── Makefile
```

## Key Features

- **Bridge Engine**: 92 initial bridges → 46 compressed via spectral analysis
- **Diffusion**: 512-dim latent, 1000 timesteps, MLP denoiser (SiLU activations)
- **Tiled Rendering**: 64×64 tiles, streaming to disk (low VRAM)
- **Large Model**: 30,000 bridges with on-demand VRAM streaming
- **Training Bot**: Auto-fetches images, augments, trains diffusion
- **Quantization**: INT8/INT4 bridge quantization support
- **NSFW Filter**: Keyword-based prompt blocking + pixel-analysis image flagging + block-blur sanitization

## Kaggle Notes

- Kaggle provides 30GB VRAM (T4/P100) - plenty for this CPU-only project
- Build takes ~30s, generation ~10-30s depending on mode
- Output raw files are 512×512×4×1byte = 1MB each
- For GPU acceleration, the CUDA diffusion code would need to be enabled
- NSFW filter runs entirely on CPU, minimal overhead